# Revisión del cruce Cochilco–SEA

Este notebook valida primero los alias empresariales y luego aplica tres reglas conservadoras antes de la revisión humana:

- `regla_auto`: confirma el mejor expediente si `score >= 95` y la empresa coincide.
- `regla_reingreso`: para nombres normalizados repetidos con `score >= 95`, elige el ingreso más reciente y conserva los expedientes anteriores en `historial`.
- `regla_sin_expediente`: asigna `NA` si el mejor puntaje es `< 75` y la empresa no coincide.
- El resto se muestra con sus tres mejores candidatos para una decisión explícita de J.

## Alias por verificar

Solo se muestran alias con estado `pendiente`. J puede cambiarlos a `confirmado` o `rechazado` en la celda siguiente.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

from cmip.match import apply_alias_decisions, build_match_review
from cmip.portfolio_status import (
    apply_automatic_match_rules,
    apply_j_decisions,
    build_confirmed_match_table,
    build_doubtful_cases,
)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ALIASES_PATH = ROOT / 'data' / 'curated' / 'company_aliases.csv'
REVIEW_PATH = ROOT / 'docs' / 'match_review.csv'
SEA_DATABASE_PATH = ROOT / 'data' / 'processed' / 'cmip.duckdb'
with duckdb.connect(str(SEA_DATABASE_PATH), read_only=True) as connection:
    sea_mining = connection.execute('SELECT * FROM sea_mining').fetchdf()
aliases = pd.read_csv(ALIASES_PATH)
display(aliases.loc[aliases['estado'].eq('pendiente')])

In [ ]:
ALIAS_J = {
    # ('Capstone Copper', 'Minera Mantoverde'): 'confirmado',
    # ('Sociedad Punta del Cobre S.A.', 'Pucobre'): 'rechazado',
}

aliases = apply_alias_decisions(aliases, ALIAS_J)
aliases.to_csv(ALIASES_PATH, index=False)
build_match_review()
print('Distribución de alias:')
display(aliases['estado'].value_counts().rename('n').to_frame())

## Aplicación de reglas automáticas

In [ ]:
review = pd.read_csv(
    REVIEW_PATH,
    dtype={'match_confirmado': 'string', 'criterio': 'string'},
    keep_default_na=False,
)
print(f'{review.cochilco_id.nunique()} proyectos; {len(review)} filas candidatas')

In [ ]:
review = apply_automatic_match_rules(review)
automaticas = review.loc[review['criterio'].ne(''), 'criterio'].value_counts()
print(f'Decisiones automáticas: {int(automaticas.sum())}')
display(automaticas.rename('n').to_frame())

## Casos dudosos

La tabla siguiente contiene los tres mejores candidatos, sus nombres, empresas, estado, fecha, puntaje, margen y señal de coincidencia empresarial.

In [ ]:
dudosos = build_doubtful_cases(review, top_n=3)
print(f'Casos pendientes: {dudosos.cochilco_id.nunique()}')
display(dudosos)

## Decisiones de J

Complete una entrada por cada `cochilco_id` dudoso. Use el `exp_id` elegido o la cadena `NA`. No incluya proyectos ya resueltos por reglas automáticas.

In [ ]:
DECISIONES_J = {
       "antofagasta-minerals-proyecto-de-adaptacion-operacional-pao": 2151393993,
       "antofagasta-minerals-extension-de-vida-util-con-transicion-hidrica": 2159410455,
       "freeport-mcmoran-modificacion-pila-de-lixiviacion-sulfolix": 2163859477,
       "marimaca-copper-corp-proyecto-minero-de-cobre-oxidos-marimaca": 2163952036,
       "rio2-ltd-fenix-gold": 2146327395,
       "nittetsu-mining-co-proyecto-minero-arqueros-cobre": 2146624103,
       "haldeman-mining-company-s-a-plan-de-desarrollo-michilla": 2154794710,
       "cementos-biobio-extension-el-way-este": 2158996386,
       "kinross-gold-corporation-lobo-marte": 2168238560,
       "antofagasta-minerals-desarrollo-minera-centinela-2-concentradora-fase-i-y-ii": 2130502645,
       "codelco-sulfuros-rt-fase-ii-3": 8210090,
       "compania-minera-dona-ines-de-collahuasi-scm-collahuasi-210-ktpd-proyecto-c20": 2142210954,
       "compania-minera-dona-ines-de-collahuasi-scm-proyecto-4a-linea-nueva-conentradora-en-rosario": "NA",  # noqa: E501
       "codelco-chuquicamata-subterranea-1": "NA",
       "bhp-reapertura-cerro-colorado": "NA",
       "bhp-aplicacion-de-ripios-de-lixiviacion-de-bhp-en-escondida": "NA",
       "capstone-copper-desarrollo-mantoverde-fase-ii": "NA",
       "bhp-crecimiento-concentradora-spence": "NA",
       "capstone-copper-mv-optimized-mv-o": "NA",
       "sqm-extension-vida-util-proyecto-maria-elena": "NA",
}

review = apply_j_decisions(review, DECISIONES_J, sea_mining=sea_mining)

In [ ]:
# Persiste tanto la decisión como su procedencia. El CSV permanece local.
review.to_csv(REVIEW_PATH, index=False)
pendientes = build_doubtful_cases(review, top_n=3)
if pendientes.empty:
    confirmed = build_confirmed_match_table(review)
    assert len(confirmed) == 59
    print('Cruce completo: 59/59 decisiones. Ya puede ejecutar `make portfolio`.')
    display(confirmed['criterio'].value_counts().rename('n').to_frame())
else:
    print(
        f'Cruce parcial guardado: faltan {pendientes.cochilco_id.nunique()} decisiones de J.'
    )